# Colab GPU Training Notebook

Use this notebook in Colab to train the multi-head 3D SE-ResNet50 model on GPU.

In [ ]:
import os
import sys

# Mount Google Drive and set paths
from google.colab import drive
drive.mount('/content/drive')

ROOT_DIR = '/content/LungInsight'
DRIVE_DIR = '/content/drive/MyDrive/lunginsight'
os.makedirs(ROOT_DIR, exist_ok=True)
os.makedirs(DRIVE_DIR, exist_ok=True)

# Copy repository files into Colab workspace if not already present
if not os.path.exists(ROOT_DIR):
    raise RuntimeError('Please upload repository content to /content/LungInsight')

sys.path.append(ROOT_DIR)

import torch
from torch.utils.data import DataLoader
from torch.optim import Adam
from cir_multihead_pipeline import create_multihead_model, FEATURE_NAMES, ExtendedCirDataset
from train_colab_gpu import build_gradnorm_model, train_one_epoch, validate_one_epoch

In [ ]:
!pip install torch torchvision numpy pandas scikit-learn pylidc gradnorm-pytorch pytorch-grad-cam

In [ ]:
TRAIN_CSV = '/content/drive/MyDrive/lunginsight/cpu_split/train_split.csv'
VAL_CSV = '/content/drive/MyDrive/lunginsight/cpu_split/val_split.csv'
BATCH_SIZE = 4
EPOCHS = 10
LR = 1e-4
ALPHA = 1.0
NUM_WORKERS = 4

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

train_dataset = ExtendedCirDataset(TRAIN_CSV, device='cpu')
val_dataset = ExtendedCirDataset(VAL_CSV, device='cpu')
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

model, gradnorm = build_gradnorm_model(device=device, alpha=ALPHA)
optimizer = Adam(model.parameters(), lr=LR)

best_val_loss = float('inf')
best_model_path = os.path.join(DRIVE_DIR, 'best_model_gpu.pth')
os.makedirs(DRIVE_DIR, exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    print(f'=== Epoch {epoch}/{EPOCHS} ===')
    train_loss = train_one_epoch(train_loader, model, gradnorm, optimizer, device)
    val_loss = validate_one_epoch(val_loader, model, device)

    print('Training losses:', train_loss)
    print('Validation losses:', val_loss)

    avg_val = sum(val_loss.values()) / len(val_loss)
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), best_model_path)
        print(f'Saved best model to {best_model_path} (avg val loss {best_val_loss:.4f})')